# Arabic OCR Post-Correction — Colab TrainingFinetunes **Qwen2.5-0.5B-Instruct** to repair Arabic OCR output.Everything (dataset, weights, checkpoints) lives on Colab's disk, not yours.The only artifact that leaves is the LoRA adapter, pushed to the HuggingFace Hub.**Before you run:** Runtime → Change runtime type → **T4 GPU**.Repo: https://github.com/Crypto47/arabic-ocr-correct

## 1. Environment

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# Colab ships torch already; only add what's missing.!pip -q install "transformers>=4.44" "peft>=0.14" "trl>=0.20" \                "datasets>=2.20" "accelerate>=0.33" pyyaml kaggleprint("deps installed")

In [ ]:
!git clone -q https://github.com/Crypto47/arabic-ocr-correct.git%cd arabic-ocr-correct!ls

## 2. Kaggle credentialsGet `kaggle.json` from your Kaggle account page → Settings → API → *Create New Token*.Run the cell and upload it.

In [ ]:
import os, jsonfrom google.colab import filesif not os.path.exists("/root/.kaggle/kaggle.json"):    print("Upload your kaggle.json:")    up = files.upload()    os.makedirs("/root/.kaggle", exist_ok=True)    with open("/root/.kaggle/kaggle.json", "wb") as f:        f.write(next(iter(up.values())))    os.chmod("/root/.kaggle/kaggle.json", 0o600)print("kaggle credentials ready")

## 3. Download the clean Arabic corpusSet `DATASET` to whichever Kaggle corpus you picked. The builder auto-detects`.txt` / `.jsonl` / `.json` / `.csv` / `.tsv`, so you do not need to know theinternal layout in advance.Candidates:| ref | size | why ||---|---|---|| `azharhasannsaif/arabic-official-documents` | 772 MB | closest domain match — real official documents || `abedkhooli/arabic-bert-corpus` | 1.7 GB | clean MSA, high volume || `z3rocool/arabic-wikipedia-dump-2021` | 419 MB | broad topic coverage || `antcorpus/antcorpus` | 10 MB | tiny — use for a fast first pass |

In [ ]:
DATASET = "antcorpus/antcorpus"   # <-- swap for your pick; start small to smoke-test!kaggle datasets download -d {DATASET} -p data/raw --unzip -q!du -sh data/raw && find data/raw -type f | head -20

## 4. Build (noisy → clean) training pairs

In [ ]:
# Corruption severity is sampled per example between --min-rate and --max-rate,# so the model sees clean scans and badly degraded ones in the same run.!python src/build_dataset.py \    --input data/raw \    --output data/processed \    --max-pairs 60000 \    --min-rate 0.04 --max-rate 0.18

In [ ]:
import jsonwith open("data/processed/train.jsonl", encoding="utf-8") as f:    for i, line in enumerate(f):        if i >= 3:            break        r = json.loads(line)        print("NOISY:", r["messages"][0]["content"].split("\n\n")[-1])        print("CLEAN:", r["messages"][1]["content"])        print("rate :", r["noise_rate"], "\n")

## 5. Baseline — score the *untuned* model firstDo this before training. If you cannot state what the base model scored, youcannot claim the finetune did anything.

In [ ]:
!python src/evaluate.py --limit 200 --out results/eval_base.json

## 6. Train

In [ ]:
!python src/train.py --config configs/qwen05b_lora.yaml

## 7. Score the finetuned model

In [ ]:
!python src/evaluate.py \    --adapter outputs/arabic-ocr-correct/final \    --limit 200 \    --out results/eval_tuned.json

In [ ]:
import jsonbase = json.load(open("results/eval_base.json", encoding="utf-8"))tuned = json.load(open("results/eval_tuned.json", encoding="utf-8"))print(f"{'':<20}{'CER':>10}{'WER':>10}")print("-" * 40)print(f"{'raw OCR':<20}{base['baseline']['cer']:>10.4f}{base['baseline']['wer']:>10.4f}")print(f"{'base 0.5B':<20}{base['corrected']['cer']:>10.4f}{base['corrected']['wer']:>10.4f}")print(f"{'finetuned':<20}{tuned['corrected']['cer']:>10.4f}{tuned['corrected']['wer']:>10.4f}")print("-" * 40)print(f"CER reduction vs raw OCR: {tuned['cer_reduction_pct']:.1f}%")print("\nPaste these numbers into the README results table.")

## 8. Try it

In [ ]:
!python src/infer.py \    --adapter outputs/arabic-ocr-correct/final \    --text "العلم نور يضنء طزيق الإنسان في الحتياة، وآلجهل ظلام دام"

## 9. Publish the adapter to HuggingFaceThe adapter is a few tens of MB — the base model is pulled from the Hub at loadtime, so nothing large is ever stored.

In [ ]:
from huggingface_hub import notebook_loginnotebook_login()

In [ ]:
from peft import PeftModelfrom transformers import AutoModelForCausalLM, AutoTokenizerREPO_ID = "Crypto47/arabic-ocr-correct-0.5b"BASE = "Qwen/Qwen2.5-0.5B-Instruct"ADAPTER = "outputs/arabic-ocr-correct/final"model = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(BASE), ADAPTER)model.push_to_hub(REPO_ID)AutoTokenizer.from_pretrained(ADAPTER).push_to_hub(REPO_ID)print(f"https://huggingface.co/{REPO_ID}")

---**Last step, and skipping it wastes the whole run:** copy the numbers fromsection 7 into the README results table and push. The repo is the artifactpeople read; the notebook is just how it got made.